<a href="https://colab.research.google.com/github/guilhermelaviola/IntegrativePracticeOfDataScienceSolutionsWithDisruptiveTechnologies/blob/main/Class16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Implementation of Projects with Computer Vision & AI**
This practical exercise proposes an intergalactic simulation in which the user acts as the chief botanist of the starship "Flower of the Galaxy" orbiting the planet Kepler-452b, with the mission of automatically cataloging the local flora. To make the project feasible, the text details the development of a complete pipeline using Python, OpenCV, and TensorFlow/Keras, structured in steps that include the acquisition of an image bank, preprocessing (such as grayscale conversion and contrast enhancement), visual feature extraction, and the training of a Convolutional Neural Network (CNN). In addition to guiding the student in the practical implementation and classification testing with new images, the material fosters reflection on the technical challenges involved and expands the learning horizon by contextualizing the use of these technologies in real-world applications, such as precision agriculture, medical diagnosis, and environmental monitoring.

In [ ]:
# Importing all the necessary libraries and resources:
import cv2
import numpy as np
import os
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

## **Example: Plant Classifier**
The following example shows a computer vision and artificial intelligence system to classify alien plants. To do this, we will build a pipeline that goes from image pre-processing to the implementation of a convolutional neural network (CNN) for visual pattern recognition. The idea is to assist the "Flower of the Galaxy" spacecraft expedition in cataloging the local flora of Kepler-452b.

In [ ]:
# Conversion to Grayscale and Contrast Enhancement:
def preprocess_image(image_path, size=(128, 128)):
  # Loading the image:
  image = cv2.imread(image_path)
  # Resizing the image:
  image = cv2.resize(image, size)
  # Converting to grayscale:
  gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
  # Enhancing contrast (histogram equalization):
  enhanced = cv2.equalizeHist(gray)
  # Removing noise with a Gaussian filter:
  denoised = cv2.GaussianBlur(enhanced, (5, 5), 0)
  return denoised

# Example of image preprocessing:
sample_image = preprocess_image('dataset/class1/example1.jpg')
plt.imshow(sample_image, cmap='gray')
plt.title('Pre-processed Image')
plt.axis('off')
plt.show()

In [ ]:
# Edge Detection with Canny:
def extract_edges(image):
  # Applying Canny's edge detector:
  edges = cv2.Canny(image, threshold1=50, threshold2=150)
  return edges

edges_image = extract_edges(sample_image)
plt.imshow(edges_image, cmap='gray')
plt.title('Edges Detected')
plt.axis('off')
plt.show()

In [ ]:
# Training and validation directories:
train_dir = 'dataset/train'
validation_dir = 'dataset/validation'

# Creating data generators with rescale and grayscale conversion:
train_datagen = ImageDataGenerator(rescale=1./255)
validation_datagen = ImageDataGenerator(rescale=1./255)

# Generator for the training set:
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(128, 128),
    color_mode='grayscale',
    batch_size=32,
    class_mode='categorical'
    )

# Generator for the validation set:
validation_generator = validation_datagen.flow_from_directory(
    validation_dir,
    target_size=(128, 128),
    color_mode='grayscale',
    batch_size=32,
    class_mode='categorical'
    )

In [ ]:
# CNN implementation:
num_classes = len(train_generator.class_indices)

model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(128, 128, 1)),
    MaxPooling2D(pool_size=(2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
    ])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
# Training the model:
history = model.fit(
    train_generator,
    epochs=20,
    validation_data=validation_generator
    )

In [ ]:
# Classification and Evaluation:
def predict_image(image_path):
  # Preprocessing the image and normalizes the values:
  image = preprocess_image(image_path)
  image = cv2.resize(image, (128, 128))
  image = image.astype('float32') / 255.0
  # Expanding the dimensions to fit the CNN input:
  image = np.expand_dims(image, axis=-1)
  image = np.expand_dims(image, axis=0)
  # Performing the prediction:
  prediction = model.predict(image)
  class_idx = np.argmax(prediction)
  return class_idx, prediction

# Example of prediction with a new image:
class_idx, prediction = predict_image('new_images/new_example.jpg')
print('Predicted class:', class_idx)